In [1]:
from tensorflow.keras.models import Model,load_model
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from sklearn.metrics import classification_report,confusion_matrix
import clahe

I0000 00:00:1788275820.347007   12434 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788275820.486978   12434 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788275822.659226   12434 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
train_path = "../Datasets/hand/Train"
test_path = "../Datasets/hand/Test"
valid_path = "../Datasets/hand/valid"

In [3]:
train_aug = ImageDataGenerator(
    preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input),
    rotation_range = 10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=0.05,
)
valid_aug = ImageDataGenerator(
    preprocessing_function=lambda img: clahe.clahe_preprocess(img, preprocess_input),
)
test_aug = ImageDataGenerator(preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input))

In [4]:
train_set = train_aug.flow_from_directory(
    train_path,
    target_size = (224,224),
    batch_size=6,
    class_mode="binary",
)
validation_set = train_aug.flow_from_directory(
    valid_path,
    target_size=(224, 224),
    batch_size=6,
    class_mode="binary",
)
test_set = test_aug.flow_from_directory(
    test_path,
    target_size = (224,224),
    batch_size=6,
    class_mode="binary",
    shuffle = False
)

Found 4241 images belonging to 2 classes.
Found 478 images belonging to 2 classes.
Found 713 images belonging to 2 classes.


In [5]:
base_model = DenseNet201(include_top=False,weights="imagenet",input_shape=(224,224,3),pooling='avg')

I0000 00:00:1788275827.544050   12434 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
base_model.trainable = False

In [7]:
x = base_model.output
x = Dense(32,activation='relu')(x)
output = Dense(1,activation='sigmoid')(x)

In [8]:
model = Model(inputs = base_model.input,outputs = output)

In [9]:
model.compile(optimizer= Adam(1e-3),loss="binary_crossentropy",metrics=['accuracy'])

In [10]:
erly_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [11]:
class_weight = {0:2, 1:1}

In [12]:
model.fit(train_set,validation_data=validation_set,epochs=30,callbacks=erly_stop,class_weight=class_weight)

Epoch 1/30


I0000 00:00:1788275833.168681   12434 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1788275844.288513   12636 service.cc:153] XLA service 0x7da358032230 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788275844.288594   12636 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1788275844.882280   12636 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788275849.838451   12636 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1788275850.054192   12636 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_34019__.432
I0000 00:00:1788275882.567982   12636 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 82/707 ━━━━━━━━━━━━━━━━━━━━ 34s 55ms/step - accuracy: 0.5468 - loss: 1.0251

I0000 00:00:1788275892.215194   12632 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_34019__.432


707/707 ━━━━━━━━━━━━━━━━━━━━ 187s 194ms/step - accuracy: 0.5834 - loss: 0.8703 - val_accuracy: 0.6423 - val_loss: 0.6133
Epoch 2/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 48s 67ms/step - accuracy: 0.6539 - loss: 0.7900 - val_accuracy: 0.5481 - val_loss: 0.7667
Epoch 3/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 56s 80ms/step - accuracy: 0.6671 - loss: 0.7756 - val_accuracy: 0.7113 - val_loss: 0.5551
Epoch 4/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 60s 85ms/step - accuracy: 0.6949 - loss: 0.7385 - val_accuracy: 0.7092 - val_loss: 0.5721
Epoch 5/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 50s 70ms/step - accuracy: 0.7100 - loss: 0.7207 - val_accuracy: 0.6925 - val_loss: 0.5712
Epoch 6/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 49s 69ms/step - accuracy: 0.7112 - loss: 0.7157 - val_accuracy: 0.6841 - val_loss: 0.5802
Epoch 7/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 59s 83ms/step - accuracy: 0.7140 - loss: 0.7119 - val_accuracy: 0.7155 - val_loss: 0.5402
Epoch 8/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 57s 80ms/step - accuracy: 0.7199 - loss: 0.6929 - val_accur

In [13]:
prob = model.predict(test_set)

119/119 ━━━━━━━━━━━━━━━━━━━━ 39s 184ms/step


In [14]:
pred = (prob >= 0.5).astype(int)

In [15]:
true_data = test_set.classes
model1 = model

In [16]:
print(classification_report(true_data,pred))
print(confusion_matrix(true_data,pred))

              precision    recall  f1-score   support

           0       0.81      0.74      0.77       342
           1       0.78      0.84      0.81       371

    accuracy                           0.79       713
   macro avg       0.79      0.79      0.79       713
weighted avg       0.79      0.79      0.79       713

[[253  89]
 [ 59 312]]


In [17]:
base_model.trainable = True

In [18]:
for layer in base_model.layers[:-80]:
    layer.trainable = False

In [19]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

In [20]:
model.compile(optimizer=Adam(1e-5),loss="binary_crossentropy",metrics=['accuracy'])

In [21]:
model.fit(train_set,validation_data=validation_set,epochs=30,callbacks=[early_stop,reduce_lr],class_weight=class_weight)

Epoch 1/30


I0000 00:00:1788276844.881682   12635 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_158380__.524


101/707 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.7353 - loss: 0.6363

I0000 00:00:1788276896.377920   12635 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_158380__.524


707/707 ━━━━━━━━━━━━━━━━━━━━ 181s 175ms/step - accuracy: 0.7104 - loss: 0.7376 - val_accuracy: 0.7259 - val_loss: 0.5746 - learning_rate: 1.0000e-05
Epoch 2/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 45s 63ms/step - accuracy: 0.7213 - loss: 0.7143 - val_accuracy: 0.7218 - val_loss: 0.5768 - learning_rate: 1.0000e-05
Epoch 3/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 58s 81ms/step - accuracy: 0.7277 - loss: 0.6954 - val_accuracy: 0.7050 - val_loss: 0.5761 - learning_rate: 1.0000e-05
Epoch 4/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 46s 69ms/step - accuracy: 0.7463 - loss: 0.6673 - val_accuracy: 0.7238 - val_loss: 0.5393 - learning_rate: 2.0000e-06
Epoch 5/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 50s 71ms/step - accuracy: 0.7475 - loss: 0.6661 - val_accuracy: 0.7050 - val_loss: 0.5599 - learning_rate: 2.0000e-06
Epoch 6/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 55s 78ms/step - accuracy: 0.7522 - loss: 0.6609 - val_accuracy: 0.6925 - val_loss: 0.5644 - learning_rate: 2.0000e-06
Epoch 7/30
707/707 ━━━━━━━━━━━━━━━━━━━━ 51s 71ms/step - accur

In [22]:
prob = model.predict(test_set)

119/119 ━━━━━━━━━━━━━━━━━━━━ 40s 198ms/step


In [23]:
pred = (prob >= 0.5).astype(int)

In [24]:
true_data = test_set.classes

In [25]:
print(classification_report(true_data,pred))
print(confusion_matrix(true_data,pred))

              precision    recall  f1-score   support

           0       0.77      0.78      0.78       342
           1       0.80      0.79      0.79       371

    accuracy                           0.79       713
   macro avg       0.79      0.79      0.79       713
weighted avg       0.79      0.79      0.79       713

[[268  74]
 [ 78 293]]


In [ ]:
moh = model

SyntaxError: invalid syntax (1289042906.py, line 1)

In [ ]:
moh.save('hand_model.keras')

In [ ]:
nov_prob = moh.predict(test_set)

119/119 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step


In [ ]:
pred_mo = (nov_prob >= 0.5).astype(int)

In [ ]:
print(classification_report(true_data,pred_mo))
print(confusion_matrix(true_data,pred_mo))

              precision    recall  f1-score   support

           0       0.83      0.77      0.80       342
           1       0.80      0.86      0.83       371

    accuracy                           0.81       713
   macro avg       0.82      0.81      0.81       713
weighted avg       0.81      0.81      0.81       713

[[262  80]
 [ 53 318]]
